# 013 — CRAG: Corrective RAG
**Generation Series** | Self-correcting retrieval loop

**CRAG** evaluates retrieved docs and corrects when they are irrelevant:
1. Retrieve → Grade relevance
2. If **CORRECT** → generate directly
3. If **AMBIGUOUS** → supplement with web search
4. If **INCORRECT** → discard, search web, generate from web results
5. Knowledge refinement: strip boilerplate, extract key facts


In [1]:
import os, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message=".*LangChain.*")
warnings.filterwarnings('ignore', message=".*allowed_objects.*")
warnings.filterwarnings('ignore', module="langchain.*")
warnings.filterwarnings('ignore', module="langgraph.*")

from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("✓ API key loaded")


✓ API key loaded


In [2]:
import os, time
os.environ["ANONYMIZED_TELEMETRY"] = "False"  # silence ChromaDB telemetry

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_retries=2)
llm_smart = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_retries=2)
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    cache_folder=os.path.join(os.getcwd(), '..', 'datasets', 'models'),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
print(f"✓ llm       : {llm.model_name}")
print(f"✓ llm_smart : {llm_smart.model_name}")
print(f"✓ embeddings: all-MiniLM-L6-v2 (384-dim, normalized cosine)")


✓ llm       : llama-3.1-8b-instant
✓ llm_smart : llama-3.3-70b-versatile
✓ embeddings: all-MiniLM-L6-v2 (384-dim, normalized cosine)


In [3]:
import os

DS_DIR = os.path.join(os.getcwd(), '..', 'datasets')
os.makedirs(DS_DIR, exist_ok=True)

def load_doc(fname):
    path = os.path.join(DS_DIR, fname)
    if os.path.exists(path):
        return open(path, encoding='utf-8').read()
    raise FileNotFoundError(
        f"Missing: {path}\n"
        "Run  python create_datasets.py  from the project root."
    )

ai_text  = load_doc("ai.txt")
ml_text  = load_doc("machine_learning.txt")
nlp_text = load_doc("nlp.txt")
dl_text  = load_doc("deep_learning.txt")
rag_text = load_doc("rag.txt")
all_text = "\n\n".join([ai_text, ml_text, nlp_text, dl_text, rag_text])

print(f"✓ 5 dataset files loaded")
print(f"  {'File':<20} {'Chars':>8}  {'~Words':>7}")
print(f"  {'-'*40}")
for name, txt in [('ai.txt', ai_text), ('machine_learning.txt', ml_text),
                  ('nlp.txt', nlp_text), ('deep_learning.txt', dl_text),
                  ('rag.txt', rag_text)]:
    print(f"  {name:<20} {len(txt):>8,}  {len(txt.split()):>7,}")
print(f"  {'-'*40}")
print(f"  {'TOTAL':<20} {len(all_text):>8,}  {len(all_text.split()):>7,}")


✓ 5 dataset files loaded
  File                    Chars   ~Words
  ----------------------------------------
  ai.txt                  6,221      943
  machine_learning.txt    5,953      870
  nlp.txt                 5,160      693
  deep_learning.txt       5,114      686
  rag.txt                 5,146      715
  ----------------------------------------
  TOTAL                  27,602    3,907


In [4]:
# ── Build local retriever ────────────────────────────────────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS

splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=50)
docs = [Document(page_content=c)
        for text in [ai_text, ml_text, dl_text, rag_text]
        for c in splitter.split_text(text[:15000])]
vectorstore = FAISS.from_documents(docs, embeddings)
retriever   = vectorstore.as_retriever(search_kwargs={"k": 4})
print(f"✓ Local retriever: {len(docs)} docs")


✓ Local retriever: 60 docs


In [5]:
# ── Document relevance grader ─────────────────────────────────────────────
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

grade_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "Grade how relevant this document is to answering the question.\n"
        "Respond with exactly one word: 'correct', 'ambiguous', or 'incorrect'."
    )),
    ("human", "Document:\n{document}\n\nQuestion: {question}"),
])
doc_grader = grade_prompt | llm | StrOutputParser()

# Web search simulator (returns plausible Wikipedia-style snippets)
def simulated_web_search(query: str) -> list:
    prompt = ChatPromptTemplate.from_template(
        "Provide 2 short factual passages (3 sentences each) that would answer: {query}\n"
        "Write them as if from a reference article. Separate with ---"
    )
    raw = (prompt | llm | StrOutputParser()).invoke({"query": query})
    snippets = [s.strip() for s in raw.split("---") if len(s.strip()) > 30]
    return [Document(page_content=s, metadata={"source": "web"}) for s in snippets]

print("✓ Grader and web-search simulator ready")


✓ Grader and web-search simulator ready


In [6]:
# ── CRAG State & Logic ────────────────────────────────────────────────────
from typing import TypedDict, List
from langgraph.graph import StateGraph, END

class CRAGState(TypedDict):
    question:  str
    documents: List[str]
    sources:   List[str]
    answer:    str

def retrieve_node(state: CRAGState):
    docs = retriever.invoke(state["question"])
    return {**state,
            "documents": [d.page_content for d in docs],
            "sources":   ["local"] * len(docs)}

def grade_and_correct(state: CRAGState):
    q       = state["question"]
    kept, sources = [], []
    verdicts = []
    for i, doc in enumerate(state["documents"]):
        v = doc_grader.invoke({"document": doc[:600], "question": q}).lower().strip()
        verdicts.append(v)
        if "correct" in v:
            kept.append(doc)
            sources.append("local")
        print(f"  Doc {i+1}: {v}")

    overall = (
        "correct"   if all("correct" in v for v in verdicts) else
        "incorrect" if all("incorrect" in v for v in verdicts) else
        "ambiguous"
    )
    print(f"  Overall: {overall}")

    if overall in ("incorrect", "ambiguous"):
        web_docs = simulated_web_search(q)
        kept   += [d.page_content for d in web_docs]
        sources += ["web"] * len(web_docs)
        print(f"  Added {len(web_docs)} web results")

    return {**state, "documents": kept, "sources": sources}

def generate_node(state: CRAGState):
    ctx = "\n\n".join(state["documents"][:4])
    chain = (
        ChatPromptTemplate.from_template(
            "Answer based only on context.\nContext:\n{ctx}\nQ: {q}\nA:"
        ) | llm | StrOutputParser()
    )
    ans = chain.invoke({"ctx": ctx[:3500], "q": state["question"]})
    return {**state, "answer": ans}

graph = StateGraph(CRAGState)
graph.add_node("retrieve",          retrieve_node)
graph.add_node("grade_and_correct", grade_and_correct)
graph.add_node("generate",          generate_node)
graph.set_entry_point("retrieve")
graph.add_edge("retrieve",          "grade_and_correct")
graph.add_edge("grade_and_correct", "generate")
graph.add_edge("generate",          END)
crag_app = graph.compile()
print("✓ CRAG graph compiled")


✓ CRAG graph compiled


/home/saurabh/miniconda3/lib/python3.13/site-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [7]:
# ── Test CRAG ─────────────────────────────────────────────────────────────
questions = [
    "What is retrieval-augmented generation?",
    "Who invented the first computer?",
]
for q in questions:
    print(f"\nQ: {q}")
    result = crag_app.invoke({"question": q, "documents": [], "sources": [], "answer": ""})
    print(f"Sources used: {list(set(result['sources']))}")
    print(f"Answer: {result['answer'][:350]}")



Q: What is retrieval-augmented generation?


  Doc 1: correct
  Doc 2: correct


  Doc 3: correct


  Doc 4: correct
  Overall: correct
Sources used: ['local']
Answer: Retrieval-Augmented Generation (RAG) is a hybrid AI architecture that combines information retrieval with text generation.

Q: Who invented the first computer?


  Doc 1: incorrect
  Doc 2: incorrect


  Doc 3: incorrect
  Doc 4: incorrect
  Overall: correct


Sources used: ['local']
Answer: There is no information in the provided context about who invented the first computer. The context is focused on the history of AI research and its developments.


## Key Takeaways
- CRAG **grades before generating** — no hallucination from bad context
- The web-search fallback ensures questions outside the local corpus can still be answered
- Knowledge refinement (stripping boilerplate) improves context signal-to-noise
- Pair CRAG with Self-RAG (notebook 010) for maximum self-correction capability
